# Completed: EDA for E-commerce transaction signals

Data Cleaning & Type Correction
Before running analytical loops, you must ensure data types are optimized. The e-commerce dataset stores dates as strings, which prevents arithmetic calculation
•	Handle missing values (impute or drop, with justification).
•	Remove duplicates.
•	Correct data types.

In [1]:
import pandas as pd
import numpy as np

# Load Datasets
fraud_df = pd.read_csv("../data/raw/Fraud_Data.csv")
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

# 1. Type Correction
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])

# 2. Check and Remove Duplicates
print(f"Fraud duplicates: {fraud_df.duplicated().sum()}")
fraud_df.drop_duplicates(inplace=True)

# 3. Handle Missing Values
# (Both datasets are generally clean, but always wrap with an imputation strategy)
print(fraud_df.isnull().sum())

Fraud duplicates: 0
user_id           0
signup_time       0
purchase_time     0
purchase_value    0
device_id         0
source            0
browser           0
sex               0
age               0
ip_address        0
class             0
dtype: int64


Exploratory Data Analysis
    Univariate distributions of key variables.
    Bivariate relationships between features and the target.
    Quantify the class imbalance in both datasets.

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Quantify Class Imbalance
for name, df, target in [('E-commerce', fraud_df, 'class')]:
    counts = df[target].value_counts()
    pct = df[target].value_counts(normalize=True) * 100
    print(f"\n--- {name} Target Distribution ---")
    print(f"Legitimate (0): {counts[0]} ({pct[0]:.3f}%)")
    print(f"Fraudulent (1): {counts[1]} ({pct[1]:.3f}%)")

# 2. Bivariate Relationship: Purchase Value vs Class
plt.figure(figsize=(10, 5))
# Fixed Seaborn syntax
sns.boxplot(data=fraud_df, x='class', y='purchase_value', hue='class', palette='Set2', legend=False)
plt.title('E-commerce Purchase Value Distribution by Target Class')
plt.savefig('../notebooks/purchase_value_eda.png')
plt.close()


--- E-commerce Target Distribution ---
Legitimate (0): 136961 (90.635%)
Fraudulent (1): 14151 (9.365%)


Geolocation Integration (The $O(N)$ Merge Strategy)
The IpAddress_to_Country.csv maps ranges using a lower and upper bound. A standard relational cross-join scales at $O(N \times M)$ and will exhaust system memory. To solve this, convert IP strings/floats into raw 64-bit integers, sort them, and use an optimized interval-tree or an asof merge

Geolocation Integration
    Convert IP addresses to integer format.
    Merge Fraud_Data.csv with IpAddress_to_Country.csv using range-based lookup.
    Analyze fraud patterns by country.


In [3]:
# 1. Convert IPv4 to integer format (Vectorized)
def ip_to_int(df, column_name):
    # If the IP is already numeric but stored as a float/string representation
    df[column_name] = df[column_name].astype(float).astype(np.int64)
    return df

# Format the bounds inside the reference map
ip_df['lower_bound_ip_address'] = ip_df['lower_bound_ip_address'].astype(np.int64)
ip_df['upper_bound_ip_address'] = ip_df['upper_bound_ip_address'].astype(np.int64)

fraud_df = ip_to_int(fraud_df, 'ip_address')

# 2. Sort dataframes to satisfy merge_asof requirements
fraud_df = fraud_df.sort_values('ip_address')
ip_df = ip_df.sort_values('lower_bound_ip_address')

# 3. Execute range-based lookup via backward matching
merged_df = pd.merge_asof(
    fraud_df, 
    ip_df, 
    left_on='ip_address', 
    right_on='lower_bound_ip_address', 
    direction='backward'
)

# 4. Strict boundary validation boundary check
valid_mask = (merged_df['ip_address'] >= merged_df['lower_bound_ip_address']) & \
             (merged_df['ip_address'] <= merged_df['upper_bound_ip_address'])

# Nullify mismatches and replace with a standard categorical fallback flag
merged_df.loc[~valid_mask, 'country'] = 'Unknown'

# Clean up temporary operational columns
merged_df.drop(['lower_bound_ip_address', 'upper_bound_ip_address'], axis=1, inplace=True, errors='ignore')
fraud_df = merged_df